In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
%pwd

'/home/ramon/Github/mlops-wine-prediction/research'

In [3]:
os.chdir("../")
%pwd

'/home/ramon/Github/mlops-wine-prediction'

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [ ]:
from src.mlops_wine_prediction.constants import *
from src.mlops_wine_prediction.utils.common import read_yaml, create_directories,save_json

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config=self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config=ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri=os.environ.get("MLFLOW_TRACKING_URI"),


        )
        return model_evaluation_config


In [7]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

/home/ramon/Github/mlops-wine-prediction/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # Salvando métricas localmente
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # O registro de modelos não funciona com armazenamento local (file store)
            if tracking_url_type_store != "file":

                # Registra o modelo
                # Existem outras formas de usar o Model Registry, dependendo do caso de uso,
                # consulte a documentação para mais informações:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")
    


In [9]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-04-17 10:41:21,603: INFO: common: arquivo yaml: config/config.yaml carregado com sucesso]
[2026-04-17 10:41:21,624: INFO: common: arquivo yaml: params.yaml carregado com sucesso]
[2026-04-17 10:41:21,626: INFO: common: arquivo yaml: schema.yaml carregado com sucesso]
[2026-04-17 10:41:21,627: INFO: common: diretório criado em: artifacts]
[2026-04-17 10:41:21,627: INFO: common: diretório criado em: artifacts/model_evaluation]


[2026-04-17 10:41:23,049: INFO: common: arquivo json salvo em: artifacts/model_evaluation/metrics.json]


2026/04/17 10:41:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 10:41:24 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/ramon/Github/mlops-wine-prediction
2026/04/17 10:41:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/17 10:41:27 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/ramon/Github/mlops-wine-prediction
2026/04/17 10:41:27 INFO mlflow.utils.environment: Detected uv project at /home/ramon/Github/mlops-wine-prediction. Attempting to export requirements via 'uv export'.
2026/04/17 10:41:27 INFO mlflow.u

🏃 View run clumsy-wren-110 at: https://dagshub.com/ramoneirao/mlops-wine-prediction.mlflow/#/experiments/0/runs/53ad2edbb68640d38199ddd411539d28
🧪 View experiment at: https://dagshub.com/ramoneirao/mlops-wine-prediction.mlflow/#/experiments/0
